In [31]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
import sys
import os

sys.path.append(os.path.abspath(".."))

In [32]:
fichier = "../data/input/Réunion Production 2024-2025-2026.xlsx"

cols_communes = ['DATE', 'OTP', 'ST', 'CONF', 'NC', 'CAUSE NC', 'SOUS-CAUSE NC', 'CALE', 'AJOUT ACHE EN OP']
cols_2025_2026 = ['Numéro de spot']
cols_2026 = ['SENS DE TRAVAIL CONFORME', 'SENS DE TRAVAIL NON CONFORME']

df_2024 = pd.read_excel(fichier, sheet_name="2024", header=2)
df_2024 = df_2024[cols_communes].copy()
df_2024['ANNEE'] = 2024

df_2025 = pd.read_excel(fichier, sheet_name="2025", header=2)
df_2025 = df_2025[cols_communes + cols_2025_2026].copy()
df_2025['ANNEE'] = 2025

df_2026 = pd.read_excel(fichier, sheet_name="2026", header=2)
df_2026 = df_2026[cols_communes + cols_2025_2026 + cols_2026].copy()
df_2026['ANNEE'] = 2026

In [33]:
for df in [df_2024, df_2025, df_2026]:
    df['MOIS'] = df['DATE'].dt.month
    df['SEMAINE'] = df['DATE'].dt.isocalendar().week.astype('Int64')

In [ ]:
valeurs_invalides = ['OK', 'X', 'OLK', 'MANQUE PRESTA', 'ST']

mapping_doublons = {
    'SÉCURAIL': 'SECURAIL',
    'FEROMOOVE': 'FERROMOVE',
    'FEROMOVE': 'FERROMOVE',
    'CLAISSE RAIL': 'CLMTP',
    'CLAISSERAIL': 'CLMTP',
    'INFRA': 'INFRARAIL',
    'INFRA RAIL': 'INFRARAIL',
    'INFARAIL': 'INFRARAIL',
    'CTSF + SNCF': 'CTSF',
    'TIME FRET': 'TFE',
    'SNCF (SILLON)': 'SNCF',
    'A-TEAM': 'ATEAM',
    'TEAM A': 'ATEAM',
    'EIFFFAGE': 'EIFFAGE',
}

for df in [df_2024, df_2025, df_2026]:
    df['ST'] = df['ST'].str.strip().str.upper()
    df['ST'] = df['ST'].replace(mapping_doublons)
    df['ST'] = df['ST'].replace(valeurs_invalides, 'NON RENSEIGNE')
    df['ST'] = df['ST'].fillna('INTERNE')

In [35]:
df_all = pd.concat([df_2024, df_2025, df_2026], ignore_index=True)

noms_mois = ['JANVIER', 'FEVRIER', 'MARS', 'AVRIL', 'MAI', 'JUIN',
             'JUILLET', 'AOUT', 'SEPTEMBRE', 'OCTOBRE', 'NOVEMBRE', 'DECEMBRE']

# Mapping OTP → CLIENT selon l'année
mapping_client = {
    2024: {
        '701': 'TX MECA NORD',
        '702': 'GRAND PROJET',
        '705': 'GRAND PROJET',
        '707': 'GRAND PROJET',
        '703': 'ETF DIRECTION MATERIEL',
        '704': 'EXTERNE',
        '706': 'DR SUD',
    },
    2025: {
        '707': 'TX MECA NORD',
        '708': 'TX MECA NORD',
        '709': 'TX MECA NORD',
        '710': 'TX MECA NORD',
        '711': 'TX MECA NORD',
        '705': 'GRAND PROJET',
        '704': 'GRAND PROJET',
        '706': 'GRAND PROJET',
        '712': 'DR SUD',
        '701': 'CHAILLOUE',
        '703': 'ETF DIRECTION MATERIEL',
        '702': 'EXTERNE',
    },
    2026: {
        '707': 'TX MECA NORD',
        '705': 'GRAND PROJET',
        '712': 'DR SUD',
        '701': 'CHAILLOUE',
        '703': 'ETF DIRECTION MATERIEL',
        '702': 'EXTERNE',
    },
}

def assigner_client(row):
    try:
        otp = str(int(float(str(row['OTP']).strip())))
    except (ValueError, TypeError):
        otp = str(row['OTP']).strip()
    annee = row['ANNEE']
    return mapping_client.get(annee, {}).get(otp, 'NON RENSEIGNE')

df_all['CLIENT'] = df_all.apply(assigner_client, axis=1)